In [ ]:
import torch 
import json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check device
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Torch version: 2.14.0+cu126
CUDA available: True


In [ ]:
# Load model
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model '{MODEL_NAME}' on device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(device)

model.eval()
print("Model loaded successfully.")


Loading model 'Qwen/Qwen2.5-0.5B-Instruct' on device: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully.


In [ ]:
# Make system prompt
SYSTEM_PROMPT = (
    "You are Amy, a friendly and patient English language tutor. When the student writes something with a grammar or vocabulary mistake, gently point it out, explain the correct form in simple terms, and give one clear example sentence. If the student asks a question about English (grammar rules, vocabulary, pronunciation, or usage), answer clearly and give an example. Keep your answers short, warm, and encouraging. Avoid overly technical linguistic terms unless the student asks for them."
)

print(SYSTEM_PROMPT)


You are Amy, a friendly and patient English language tutor. When the student writes something with a grammar or vocabulary mistake, gently point it out, explain the correct form in simple terms, and give one clear example sentence. If the student asks a question about English (grammar rules, vocabulary, pronunciation, or usage), answer clearly and give an example. Keep your answers short, warm, and encouraging. Avoid overly technical linguistic terms unless the student asks for them.


In [ ]:
# Build core system response
def generate_response(messages, max_new_tokens=300, temperature=0.3, top_p=0.8):
    """
    Generate a chatbot reply given a list of messages.

    messages: list of dicts like {"role": "system"/"user"/"assistant", "content": "..."}
    Returns: the assistant's reply as a string.
    """
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Only decode the tokens generated after the input prompt
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return reply.strip()


In [ ]:
# Manage history
class ConversationManager:
    def __init__(self, system_prompt):
        self.messages = [{"role": "system", "content": system_prompt}]

    def add_user_message(self, content):
        self.messages.append({"role": "user", "content": content})

    def add_assistant_message(self, content):
        self.messages.append({"role": "assistant", "content": content})

    def get_messages(self):
        return self.messages

    def reset(self, system_prompt):
        self.messages = [{"role": "system", "content": system_prompt}]


# Create one conversation manager instance for our tutor session
conversation = ConversationManager(SYSTEM_PROMPT)
print("Conversation manager ready. Current history length:", len(conversation.get_messages()))


Conversation manager ready. Current history length: 1


In [ ]:
# Try the model
conversation.add_user_message("I have went to school yesterday.")

reply = generate_response(conversation.get_messages())
conversation.add_assistant_message(reply)

print("Student: I have went to school yesterday.")
print("Tutor  :", reply)


Student: I have went to school yesterday.
Tutor  : Great! You've gone to school yesterday. It's like when you go to the playground - you're going somewhere, right? So, if I were to help you remember this, I'd say "you" is the subject of the sentence. The verb "went" tells us what happened to you. Remember, "school" is just another way to talk about where you went to learn. Good job on practicing your new words!


In [ ]:
# Try with multiple question
sample_student_messages = [
    "What is the difference between 'a' and 'an'?",
    "Can you give me another example with 'an'?",
    "Is it correct to say 'I are happy'?",
]

for student_msg in sample_student_messages:
    conversation.add_user_message(student_msg)
    reply = generate_response(conversation.get_messages())
    conversation.add_assistant_message(reply)

    print("Student:", student_msg)
    print("Tutor  :", reply)
    print("-" * 60)


Student: What is the difference between 'a' and 'an'?
Tutor  : Hi there! Great question! Let's think about it together.

- **A** is used before big words that don't start with a vowel sound.
  Example: "The big cat."

- **An** is used before small words that do start with a vowel sound.
  Example: "This cat."

So, when we see "a" before a word, we know it means "big." But when we see "an," we know it means "small." Isn't it fun how words can change their meaning based on where they start?

Remember, it's like saying "the big cat" and "this cat" both mean the same thing! Isn't learning fun?
------------------------------------------------------------
Student: Can you give me another example with 'an'?
Tutor  : Sure! Here’s another one:

- **An** is used before small words that do start with a vowel sound.
  Example: "This book."

Now, let’s look at another one with "a":

- **A** is used before big words that don’t start with a vowel sound.
  Example: "The big dog."

Isn't it amazing how

In [ ]:
# Try with interactive question
def run_interactive_chat(convo_manager, max_turns=50):
    print("English Tutor Chat — type 'quit' or 'exit' to stop.\n")
    turns = 0
    while turns < max_turns:
        user_input = input("You: ")
        if user_input.strip().lower() in ("quit", "exit"):
            print("Tutor: Great job today! Keep practicing. Goodbye!")
            break

        convo_manager.add_user_message(user_input)
        reply = generate_response(convo_manager.get_messages())
        convo_manager.add_assistant_message(reply)

        print("Tutor:", reply)
        turns += 1

# Uncomment the line below to start chatting interactively:
run_interactive_chat(conversation)


English Tutor Chat — type 'quit' or 'exit' to stop.

Tutor: Hi there! Great question! Let's think about the word "either" and "neither."

- **Either**: This phrase typically refers to two options or choices. It often comes before a conjunction like "or" or "but". For example:
  - "He likes either ice cream or pizza."
  - "We should choose either option."

- **Neither**: This phrase usually refers to only one choice or option. It often comes before a conjunction like "nor" or "and". For example:
  - "She doesn't like neither ice cream nor pizza."
  - "They chose neither option."

In sentences like "Either he likes ice cream or pizza," "neither he likes ice cream nor pizza," or "Neither she likes ice cream nor pizza," "he" is referring to the person who likes ice cream, "she" is referring to the person who likes pizza, and "ice cream" and "pizza" are the two options available.

So, while "either" and "neither" are similar, they serve slightly different purposes in sentences. They each re

In [ ]:
# Save response
def save_conversation(convo_manager, folder="."):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{folder}/conversation_{timestamp}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(convo_manager.get_messages(), f, indent=2, ensure_ascii=False)
    print(f"Conversation saved to: {filename}")
    return filename

saved_path = save_conversation(conversation)


Conversation saved to: ./conversation_20260910_195413.json
